# Electra-base (Pretrained Model) 

This notebook fine-tunes `google/electra-base-discriminator`, a pretrained transformer encoder, for multiple-choice answer selection using HuggingFace's `AutoModelForMultipleChoice`.

**Approach:** each (prompt, option) pair is tokenized as a sentence pair using Electra's pretrained tokenizer and the model, which is already pretrained on general language understanding, is fine-tuned end-to-end to score the correct option highest across the 5 choices.

# Importing Libraries

In [1]:
import os
import re
import random
import numpy as np
import pandas as pd
from collections import Counter
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_cosine_schedule_with_warmup
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score
import wandb
from kaggle_secrets import UserSecretsClient
from dataclasses import dataclass, asdict

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

sns.set_theme(style="whitegrid", palette="muted")

Using device: cuda


# Loading Dataset

In [2]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
print("Datasets loaded!")

Datasets loaded!


# Define the Model

## Configuration

In [3]:
@dataclass
class Config:
    model_name: str = "google/electra-base-discriminator"
    max_seq_len: int = 150
    
    epochs: int = 5
    batch_size: int = 16
    gradient_accumulation_steps: int = 1
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1

    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    wandb_project: str = "23f2004791-t22026"
    wandb_run_name: str = "electra-base-2"
    
    def to_dict(self):
        return asdict(self)

cfg = Config()

In [4]:
try:
    user_secrets = UserSecretsClient()
    wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_api_key)
    print("Successfully logged into Weights & Biases!")
except Exception as e:
    print(f"W&B Login Failed.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: adrija935 (23f2004791-dl-genai-project) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Successfully logged into Weights & Biases!


## Tokenizer

In [5]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## PyTorch Dataset and DataLoaders

In [6]:
class MCQElectraDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256, is_test=False):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test
        self.label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row['prompt'])
        options = [str(row[opt]) for opt in ['A', 'B', 'C', 'D', 'E']]
        
        # Duplicate prompt 5 times to pair with each option
        first_sentences = [prompt] * 5
        second_sentences = options

        # Tokenize (Prompt, Option) pairs
        encoding = self.tokenizer(
            first_sentences,
            second_sentences,
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt"
        )
        
        item = {
            'input_ids': encoding['input_ids'],
            'attention_mask': encoding['attention_mask']
        }
        
        if 'token_type_ids' in encoding:
            item['token_type_ids'] = encoding['token_type_ids']
            
        if not self.is_test:
            item['label'] = torch.tensor(self.label_map[row['answer']], dtype=torch.long)
            
        return item

## Data Preprocessing and Splitting

In [7]:
train_split, val_split = train_test_split(train_df, random_state=SEED)

train_dataset = MCQElectraDataset(train_split, tokenizer, max_len=cfg.max_seq_len)
val_dataset = MCQElectraDataset(val_split, tokenizer, max_len=cfg.max_seq_len)
test_dataset = MCQElectraDataset(test_df, tokenizer, max_len=cfg.max_seq_len, is_test=True)

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=cfg.batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=False)

## Scoring Metric (MAP@3)

In [8]:
# MAP@3 Metric
def compute_map_at_3(predictions, targets):
    scores = []
    for top_preds, target in zip(predictions, targets):
        score = 0.0
        for rank, pred in enumerate(top_preds):
            if pred == target:
                score = 1.0 / (rank + 1)
                break
        scores.append(score)
    return np.mean(scores)

# Training and Validation

In [9]:
model = AutoModelForMultipleChoice.from_pretrained(cfg.model_name).to(cfg.device)

total_steps = (len(train_loader) // cfg.gradient_accumulation_steps) * cfg.epochs
warmup_steps = int(total_steps * cfg.warmup_ratio)

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForMultipleChoice LOAD REPORT from: google/electra-base-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
electra.embeddings_project.weight                 | UNEXPECTED | 
electra.embeddings_project.bias                   | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.bias                                   | MISSING    | 
sequence_summary.summary.bias                     | MISSING    | 
classifier.weight                                 | MISSING    | 
sequence_summary.summary.weight                   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	

In [10]:
wandb.init(
    project=cfg.wandb_project, 
    name=cfg.wandb_run_name, 
    config=cfg.to_dict(), 
    reinit=True
)

best_map3 = 0.0

scaler = torch.amp.GradScaler('cuda')

for epoch in range(cfg.epochs):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.epochs}")
    for step, batch in enumerate(loop):
        input_ids = batch['input_ids'].to(cfg.device)
        attention_mask = batch['attention_mask'].to(cfg.device)
        labels = batch['label'].to(cfg.device)
        
        kwargs = {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels
        }
        if 'token_type_ids' in batch:
            kwargs['token_type_ids'] = batch['token_type_ids'].to(cfg.device)
            
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            outputs = model(**kwargs)
            loss = outputs.loss / cfg.gradient_accumulation_steps
            
        scaler.scale(loss).backward()
        
        if (step + 1) % cfg.gradient_accumulation_steps == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            scaler.step(optimizer)
            scaler.update()
            
            scheduler.step()
            optimizer.zero_grad()
            
        running_loss += outputs.loss.item() * input_ids.size(0)
        
    train_loss = running_loss / len(train_loader.dataset)
    
    # Validation Loop
    model.eval()
    val_loss = 0.0
    all_top3_preds = []
    all_top1_preds = []
    all_targets = []
    
    with torch.no_grad():
        val_loop = tqdm(val_loader, desc="Validation")
        for batch in val_loop:
            input_ids = batch['input_ids'].to(cfg.device)
            attention_mask = batch['attention_mask'].to(cfg.device)
            labels = batch['label'].to(cfg.device)
            
            kwargs = {
                'input_ids': input_ids,
                'attention_mask': attention_mask,
                'labels': labels
            }
            if 'token_type_ids' in batch:
                kwargs['token_type_ids'] = batch['token_type_ids'].to(cfg.device)
                
            outputs = model(**kwargs)
            logits = outputs.logits  
            
            val_loss += outputs.loss.item() * input_ids.size(0)

            top1_preds = torch.argmax(logits, dim=1)
            _, top3_indices = torch.topk(logits, k=3, dim=1)

            all_top1_preds.append(top1_preds.cpu().numpy())
            all_top3_preds.append(top3_indices.cpu().numpy())
            all_targets.append(labels.cpu().numpy())
            
    val_loss = val_loss / len(val_loader.dataset)
    all_top1_preds = np.concatenate(all_top1_preds, axis=0)
    all_top3_preds = np.concatenate(all_top3_preds, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)
    
    val_map3 = compute_map_at_3(all_top3_preds, all_targets)
    val_accuracy = accuracy_score(all_targets, all_top1_preds)
    val_f1 = f1_score(all_targets, all_top1_preds, average='macro')

    print(f"Epoch {epoch+1}/{cfg.epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Val MAP@3: {val_map3:.4f} | Val Acc: {val_accuracy:.4f} | Val F1: {val_f1:.4f}")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_map3": val_map3,
        "val_accuracy": val_accuracy,
        "val_f1": val_f1
    })

    if val_map3 > best_map3:
        best_map3 = val_map3
        torch.save(model.state_dict(), "best_electra_model.pt")
        wandb.run.summary["best_map3"] = best_map3
        wandb.run.summary["best_accuracy"] = val_accuracy
        wandb.run.summary["best_f1"] = val_f1
        print(f"New best model saved with MAP@3: {best_map3:.4f}")

wandb.finish()

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: setting up run 8xmdq0fe
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260803_143626-8xmdq0fe
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run electra-base-2
wandb: ⭐️ View project at https://wandb.ai/23f2004791-dl-genai-project/23f2004791-t22026
wandb: 🚀 View run at https://wandb.ai/23f2004791-dl-genai-project/23f2004791-t22026/runs/8xmdq0fe

Validation: 100%|██████████| 32/32 [00:19<00:00,  1.60it/s]


Epoch 1/5 | Train Loss: 1.4967 | Val Loss: 1.0066 | Val MAP@3: 0.8480 | Val Acc: 0.7460 | Val F1: 0.7439
New best model saved with MAP@3: 0.8480


Validation: 100%|██████████| 32/32 [00:20<00:00,  1.59it/s]


Epoch 2/5 | Train Loss: 0.7587 | Val Loss: 0.3247 | Val MAP@3: 0.9737 | Val Acc: 0.9540 | Val F1: 0.9493
New best model saved with MAP@3: 0.9737


...

Validation: 100%|██████████| 32/32 [00:20<00:00,  1.55it/s]


Epoch 5/5 | Train Loss: 0.1355 | Val Loss: 0.0814 | Val MAP@3: 0.9980 | Val Acc: 0.9960 | Val F1: 0.9962


wandb: updating run metadata


New best model saved with MAP@3: 0.9980


wandb: uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb:        epoch ▁▃▅▆█
wandb:   train_loss █▄▂▁▁
wandb: val_accuracy ▁▇▇██
wandb:       val_f1 ▁▇▇██
wandb:     val_loss █▃▁▁▁
wandb:     val_map3 ▁▇███
wandb: 
wandb: Run summary:
wandb: best_accuracy 0.996
wandb:       best_f1 0.99622
wandb:     best_map3 0.998
wandb:         epoch 5
wandb:    train_loss 0.13554
wandb:  val_accuracy 0.996
wandb:        val_f1 0.99622
wandb:      val_loss 0.08144
wandb:      val_map3 0.998
wandb: 
wandb: 🚀 View run electra-base-2 at: https://wandb.ai/23f2004791-dl-genai-project/23f2004791-t22026/runs/8xmdq0fe
wandb: ⭐️ View project at: https://wandb.ai/23f2004791-dl-genai-project/23f2004791-t22026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260803_143626-8xmdq0fe/logs


# Inference

In [11]:
model.load_state_dict(torch.load("best_electra_model.pt"))
model.eval()

idx_to_label = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}
submission_preds = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(cfg.device)
        attention_mask = batch['attention_mask'].to(cfg.device)
        
        kwargs = {
            'input_ids': input_ids,
            'attention_mask': attention_mask
        }
        if 'token_type_ids' in batch:
            kwargs['token_type_ids'] = batch['token_type_ids'].to(cfg.device)
            
        outputs = model(**kwargs)
        logits = outputs.logits
        _, top3_indices = torch.topk(logits, k=3, dim=1)
        
        for top_three in top3_indices.cpu().numpy():
            pred_str = " ".join([idx_to_label[i] for i in top_three])
            submission_preds.append(pred_str)

# Generate Submission

In [12]:
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'Prediction': submission_preds
})

submission_df.to_csv('submission.csv', index=False)
print("Submission saved to 'submission.csv'!")
print(submission_df.head())

Submission saved to 'submission.csv'!
   id Prediction
0   1      A C D
1   2      B E D
2   3      B E C
3   4      E C D
4   5      C D B
